In [ ]:
import os
import sys
import json
import pandas as pd
from collections import Counter

# Setup path
ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA_DIR = os.path.join(ROOT_DIR, "data")

# Import label mapping
try:
    from extract_labels_standardized import ID_TO_LABEL
except ImportError:
    sys.path.insert(0, ROOT_DIR)
    from extract_labels_standardized import ID_TO_LABEL

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"Loaded {len(ID_TO_LABEL)} punctuation labels")

In [ ]:
# Load data from JSON files
train_all_path = os.path.join(DATA_DIR, "train_all.json")
train_all_synthetic_path = os.path.join(DATA_DIR, "train_all_synthetic.json")

print("Loading datasets...")
with open(train_all_path, 'r', encoding='utf-8') as f:
    train_all = json.load(f)

with open(train_all_synthetic_path, 'r', encoding='utf-8') as f:
    train_all_synthetic = json.load(f)

print(f"✓ train_all.json: {len(train_all)} examples loaded")
print(f"✓ train_all_synthetic.json: {len(train_all_synthetic)} examples loaded")

In [ ]:
# Analyze punctuation distribution in train_all.json
punct_counter_train_all = Counter()

for example in train_all:
    for label_id in example["ner_tags"]:
        if label_id != 0:  # Skip "O" (no punctuation)
            punct_label = ID_TO_LABEL.get(label_id, f"UNKNOWN_{label_id}")
            punct_counter_train_all[punct_label] += 1

# Get top-3 most frequent and 3 rarest
top_3_most = punct_counter_train_all.most_common(3)
top_3_rarest = punct_counter_train_all.most_common()[-3:][::-1]  # Reverse to show rarest first

print("\n=== PUNCTUATION DISTRIBUTION (train_all.json) ===")
print("\nTop-3 Most Frequent:")
for punct, count in top_3_most:
    print(f"  {punct!r}: {count}")

print("\nTop-3 Rarest:")
for punct, count in top_3_rarest:
    print(f"  {punct!r}: {count}")

print(f"\nTotal unique punctuation marks: {len(punct_counter_train_all)}")
print(f"Total punctuation occurrences: {sum(punct_counter_train_all.values())}")

In [ ]:
# Create table: Top-3 and Rarest Punctuation Signs
top_3_data = []

for rank, (punct, count) in enumerate(top_3_most, 1):
    top_3_data.append({
        "Rank": rank,
        "Punctuation Sign": punct,
        "Count": count,
        "Type": "Most Frequent"
    })

for rank, (punct, count) in enumerate(top_3_rarest, 1):
    top_3_data.append({
        "Rank": rank,
        "Punctuation Sign": punct,
        "Count": count,
        "Type": "Rarest"
    })

df_top3 = pd.DataFrame(top_3_data)

print("\n" + "="*70)
print("TABLE 1: TOP-3 MOST FREQUENT & TOP-3 RAREST PUNCTUATION SIGNS")
print("="*70)
print(df_top3.to_string(index=False))
print("="*70)

In [ ]:
# Count chunks and basic statistics
chunks_train_all = len(train_all)
chunks_train_all_synthetic = len(train_all_synthetic)
added_synthetic = chunks_train_all_synthetic - chunks_train_all

# Total tokens
total_tokens_train_all = sum(len(ex["tokens"]) for ex in train_all)
total_tokens_synthetic = sum(len(ex["tokens"]) for ex in train_all_synthetic)

# Average tokens per chunk
avg_tokens_train_all = total_tokens_train_all / chunks_train_all if chunks_train_all > 0 else 0
avg_tokens_synthetic = total_tokens_synthetic / chunks_train_all_synthetic if chunks_train_all_synthetic > 0 else 0

print("\n" + "="*70)
print("TABLE 2: CHUNK COUNT SUMMARY")
print("="*70)

chunk_summary = pd.DataFrame({
    "Dataset": ["train_all.json", "train_all_synthetic.json", "Difference (synthetic added)"],
    "Chunks": [chunks_train_all, chunks_train_all_synthetic, added_synthetic],
    "Total Tokens": [total_tokens_train_all, total_tokens_synthetic, 
                     total_tokens_synthetic - total_tokens_train_all],
    "Avg Tokens/Chunk": [f"{avg_tokens_train_all:.2f}", f"{avg_tokens_synthetic:.2f}", "-"]
})

print(chunk_summary.to_string(index=False))
print("="*70)

In [ ]:
# Count punctuation signs in both datasets
punct_counter_synthetic = Counter()

for example in train_all_synthetic:
    for label_id in example["ner_tags"]:
        if label_id != 0:  # Skip "O"
            punct_label = ID_TO_LABEL.get(label_id, f"UNKNOWN_{label_id}")
            punct_counter_synthetic[punct_label] += 1

# Create comprehensive comparison
all_puncts = set(punct_counter_train_all.keys()) | set(punct_counter_synthetic.keys())
all_puncts_sorted = sorted(all_puncts)

comparison_data = []
for punct in all_puncts_sorted:
    count_train_all = punct_counter_train_all[punct]
    count_synthetic = punct_counter_synthetic[punct]
    total_count = count_train_all + count_synthetic
    
    # Calculate percentage contribution
    pct_train_all = (count_train_all / total_count * 100) if total_count > 0 else 0
    pct_synthetic = (count_synthetic / total_count * 100) if total_count > 0 else 0
    
    comparison_data.append({
        "Punctuation": punct,
        "train_all": count_train_all,
        "synthetic": count_synthetic,
        "Total": total_count,
        "% in train_all": f"{pct_train_all:.1f}%",
        "% synthetic": f"{pct_synthetic:.1f}%"
    })

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("TABLE 3: COMPREHENSIVE PUNCTUATION STATISTICS (All Signs)")
print("="*100)
print(df_comparison.to_string(index=False))
print("="*100)

# Summary statistics
total_punct_train_all = sum(punct_counter_train_all.values())
total_punct_synthetic = sum(punct_counter_synthetic.values())
total_punct_combined = total_punct_train_all + total_punct_synthetic

print(f"\nSummary:")
print(f"  Total punctuation marks in train_all: {total_punct_train_all}")
print(f"  Total punctuation marks in synthetic: {total_punct_synthetic}")
print(f"  Combined total: {total_punct_combined}")
print(f"  Synthetic contribution: {total_punct_synthetic / total_punct_combined * 100:.2f}%")

In [ ]:
# Optional: Export tables to CSV for diploma thesis
output_dir = os.path.join(ROOT_DIR, "results")
os.makedirs(output_dir, exist_ok=True)

# Export Table 1: Top-3 and Rarest
csv_path_1 = os.path.join(output_dir, "table1_top_rarest_signs.csv")
df_top3.to_csv(csv_path_1, index=False)
print(f"✓ Exported Table 1 to: {csv_path_1}")

# Export Table 2: Chunk Summary
csv_path_2 = os.path.join(output_dir, "table2_chunk_summary.csv")
chunk_summary.to_csv(csv_path_2, index=False)
print(f"✓ Exported Table 2 to: {csv_path_2}")

# Export Table 3: Comprehensive Statistics
csv_path_3 = os.path.join(output_dir, "table3_comprehensive_statistics.csv")
df_comparison.to_csv(csv_path_3, index=False)
print(f"✓ Exported Table 3 to: {csv_path_3}")

print("\nAll statistics tables exported successfully for your diploma thesis!")

# Data Statistics: train_all.json vs train_all_synthetic.json

Analysis of punctuation distribution and dataset composition for diploma thesis.